## 03-sql

In [18]:
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
from langchain_community.utilities import SQLDatabase
import os

DB_URI = os.environ.get('DB_URI')

db = SQLDatabase.from_uri(DB_URI)

In [20]:
#print(db.get_table_info())
print(db.run('select * from sales limit 5;'))

[(1, datetime.date(2024, 1, 17), 'C021', 'P2084', '과자', '식품', 7, 9480, 66360, '정동훈', '대구'), (2, datetime.date(2024, 4, 18), 'C042', 'P8517', '음료수', '식품', 5, 2584, 12920, '이영희', '부산'), (3, datetime.date(2024, 10, 14), 'C035', 'P8019', '청소기', '생활용품', 10, 254700, 2547000, '박민수', '부산'), (4, datetime.date(2024, 3, 11), 'C033', 'P1771', '쌀', '식품', 4, 35008, 140032, '이영희', '인천'), (5, datetime.date(2024, 11, 1), 'C005', 'P8668', '음료수', '식품', 17, 2529, 42993, '정동훈', '서울')]


In [21]:
# LLM 초기화
from langchain_openai import ChatOpenAI
model = ChatOpenAI(name='gpt-4.1-mini')

In [22]:
# Agent용 Tool 만들기

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

In [24]:
# Agent 만들기
from langchain.agents import create_agent

dialect = db.dialect
top_k = 5

system_prompt = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
"""

agent = create_agent(model, toolkit.get_tools(), system_prompt=system_prompt)

for tool in toolkit.get_tools():
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [33]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    toolkit.get_tools(),
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
                interrupt_on={'sql_db_query': True},
                description_prefix='Tool 실행 전에 승인을 기다림'
        )
    ],
    checkpointer=InMemorySaver() #일시정지 - 재실행에서 돌아갈 곳을 기억해야함!

)

In [34]:
from langgraph.types import Command

question = '2월에 가장 많이 팔린 물건 3개와, 해당 물건들의 토요일 일요일 평균 매출액'

config = {'configurable': {'thread_id': '123456'}}

for event in agent.stream(
    {'messages': [{'role': 'user', 'content': question}]},
    stream_mode='values',
    config=config,
):
    if "__interrupt__" in event: 
        print("INTERRUPTED:") 
        interrupt = event["__interrupt__"][0] 
        for request in interrupt.value["action_requests"]: 
            print(request["description"]) 
    elif "messages" in event:
        event["messages"][-1].pretty_print()
    else:
        pass

print('-----------------------------------------------------------------')

for step in agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}), 
    config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()
    elif "__interrupt__" in step:
        print("INTERRUPTED:")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(request["description"])
    else:
        pass

================================ Human Message =================================

2월에 가장 많이 팔린 물건 3개와, 해당 물건들의 토요일 일요일 평균 매출액
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_wCcNiTN9lTCJWAfT2yUMkSvZ)
 Call ID: call_wCcNiTN9lTCJWAfT2yUMkSvZ
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

courses, customers, dt_demo, members, sales, students, students_courses
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_fFiSJNOh1xlP6pZsj4Oms902)
 Call ID: call_fFiSJNOh1xlP6pZsj4Oms902
  Args:
    table_names: sales
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE sales (
	id INTEGER NOT NULL, 
	order_date DATE NOT NULL, 
	customer_id VARCHAR(10) NOT NULL, 
	product_id VARCHAR(10) NOT NULL, 
	product_name VARCHAR(50) NOT NULL, 
	categ